# 02 · Preparação dos dados (v3)
**CRISP-DM:** Preparação dos Dados

1. Consolidação das chaves repetidas;
2. **Janelas de 1 h** por recurso (base do perfil normal);
3. **Períodos de 15 e 30 min** alinhados (para calibrar os limites dessas faixas contra o perfil de 1 h);
4. Mesmo período 7 dias antes (para o baseline semana anterior).

| Atributo (decisão) | Definição |
|---|---|
| `log_vol` | log(1 + transações), **na escala horária** (× 60 / minutos do período) |
| `tx_4xx` | transações 4xx / total (erros de negócio) |
| `tx_5xx` | transações 5xx / total (erros de sistema) |
| `log_lat_p95` | log(1 + **p95 da latência**): percentil 95 ponderado pelo volume das latências médias por minuto × cliente das respostas 2xx |

**Sobre o p95.** Como cada linha já é a média de um minuto × cliente, o p95 é o percentil **dessas médias** (ponderado pelo número de transações), e não das requisições individuais. Ele é consistente entre janelas e sensível a degradações que atingem só parte dos minutos ou dos clientes; o p95 real por requisição tende a ser maior.

Contexto: hora (Brasília) e tipo de dia (**útil / não útil**; feriado = não útil).

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v3) carregado · /home/wsl/gh/mestrado/claude9/v3


In [2]:
brutos = consolidar(carregar())
print(f"registros consolidados: {len(brutos):,}")
brutos.to_parquet(RESULTADOS / "02_brutos_consolidados.parquet", index=False)

registros consolidados: 5,596,868


In [3]:
jan1h = agregar(brutos, "1h")
jan1h.to_parquet(RESULTADOS / "02_janelas_60min.parquet", index=False)
for L in (15, 30):
    pL = agregar(brutos, f"{L}min")
    pL.to_parquet(RESULTADOS / f"02_janelas_{L}min.parquet", index=False)
    print(f"{L} min: {len(pL):,} períodos")
print(f"1 h: {len(jan1h):,} janelas")

15 min: 118,778 períodos


30 min: 59,389 períodos
1 h: 29,695 janelas


## Verificação: agregação por janela × agregação de período arbitrário
As duas rotinas (vetorizada, usada na calibração; e por período, usada nas consultas) devem dar o mesmo resultado para o mesmo intervalo.

In [4]:
B = Brutos(brutos)
amostra = jan1h.dropna(subset=["lat_p95"]).sample(200, random_state=0)
dif = []
for row in amostra.itertuples():
    f = B.fatia(row.recurso, row.janela, row.janela + pd.Timedelta("1h"))
    o = agregar_periodo(f, row.janela, row.janela + pd.Timedelta("1h"))
    dif.append([abs(o[a] - getattr(row, a)) for a in ATRIBUTOS])
print("maior diferença por atributo:", dict(zip(ATRIBUTOS, np.nanmax(dif, axis=0).round(9))))

maior diferença por atributo: {'log_vol': np.float64(0.0), 'tx_4xx': np.float64(0.0), 'tx_5xx': np.float64(0.0), 'log_lat_p95': np.float64(0.0)}


## Latência p95 × latência média

In [5]:
ok2 = brutos[brutos["status"] // 100 == 2].assign(janela=lambda d: d["ts"].dt.floor("1h"), tt=lambda d: d["qtd"] * d["tmp"])
media2 = ok2.groupby(["recurso", "janela"])[["tt", "qtd"]].sum().eval("tt / qtd").rename("lat_media")
comp = jan1h.set_index(["recurso", "janela"])[["lat_p95"]].join(media2)
print("razão p95 / média das respostas 2xx (mediana):", round(float((comp["lat_p95"] / comp["lat_media"]).median()), 2))
jan1h[ATRIBUTOS + ["vol", "lat_p95", "n_cli"]].describe().T.round(3)

razão p95 / média das respostas 2xx (mediana): 1.42


,count,mean,std,min,25%,50%,75%,max
log_vol,29695.0,6.994,1.377,0.000,6.159,6.848,7.844,11.767
tx_4xx,29695.0,0.175,0.309,0.000,0.001,0.009,0.188,1.000
tx_5xx,29695.0,0.003,0.018,0.000,0.000,0.000,0.000,0.821
log_lat_p95,28993.0,4.584,0.659,3.258,4.094,4.407,4.977,8.291
vol,29695.0,3255.366,8968.070,0.000,472.000,941.000,2549.000,128966.000
lat_p95,28993.0,127.926,149.066,25.000,59.000,81.000,144.000,3988.000
n_cli,29695.0,21.478,33.888,0.000,6.000,10.000,20.000,494.000


## Amostras por contexto no histórico de maio a agosto (1 h)

In [6]:
tr = recortar(jan1h, TREINO)
tr.groupby(["recurso", "tipo_dia", "hora"]).size().groupby("tipo_dia").describe()[["min", "50%", "max"]]

,min,50%,max
tipo_dia,,,
nao_util,14.0,33.5,38.0
util,32.0,77.5,85.0
